# Where does MegaDescriptor look? Similarity heatmaps

Like the attention maps in lecture ("Show, Attend and Tell"), these figures color each frame by **which regions made the baseline match a validation video to a training video**.

**Why not raw attention weights?** MegaDescriptor is a Swin transformer: it has no `[CLS]` token and its self-attention only looks inside small local windows, so there is no single attention map to read off like in a plain ViT.

**What we use instead (exact for our baseline):**
- The last Swin stage gives a **7 x 7 grid** of 768-number patch features per frame.
- The frame embedding is the **average of those 49 patch features**; the video embedding is the normalized average of its 20 frame embeddings.
- So the cosine similarity splits exactly into one number per patch: `patch feature . unit embedding of the other video`. Adding up every patch of every frame gives back the similarity.

**Only the validation split is used.** The test split stays held out for the final evaluation.

Sections 1-4 only need the saved embeddings. From section 5 on, the frames of the plotted videos must be in `data/PolarBearVidID/` (see README).

## How to read one figure

Every figure has **3 sections**, one per video, each showing **all 20 frames**:

| Section | What it is | Heatmap colored by |
|---|---|---|
| 1 | the **validation video** | what matched the section-2 video |
| 2 | the **winner**: the most similar training video (its bear is the prediction) | what matched the validation video |
| 3 | the **runner-up from the other side**: for a correct prediction, the closest video of a *wrong* bear; for a wrong prediction, the closest video of the *true* bear | what matched the validation video |

Inside each section:
- **Top left, the video-level view:** the average of the 20 frames (a ghostly overlay, since the bear moves) with the **average of the 20 heatmaps** on top. The frames are crops around the bear, so this shows which part of the crop the video's match comes from overall.
- **Top right, how the 20 frames add up:** each bar is one frame's share of the similarity. The dashed line is an equal share (5%). Bars well below it are frames that barely counted.
- **Below: all 20 frames in 4 rows x 5**, each as the **plain photo** (to judge blur, darkness, pose) with its **heatmap** right under it. The % above a photo is that frame's bar from the chart.

How to read the colors and numbers:
- **Red/yellow** = this region pushed the two videos together. **Blue** = it added little. The whole figure shares one color scale, so a frame that is blue everywhere really did count for little.
- A hot cell means *this region and its immediate surroundings*: Swin mixes each 7 x 7 cell with its neighbors (see the checks in section 5).
- **Margin** (in the title) = similarity to the true bear's best video minus similarity to the best other bear's video. Positive = correct; near 0 = a coin flip.

### Questions to ask of each figure (answers go in the notes table at the end)
1. **Bear or background?** Is the heat on the bear's body, or on rocks, water, walls? Heat on background means the match may come from the *enclosure*, not the *bear*.
2. **Which body part?** Head, back, legs, whole body? Is it the same part in row 1 and row 2?
3. **Which frames are weak?** Look at the frames with a low %: is the bear small, dark, blurry, cut off, facing away?
4. **Same place?** Do rows 1 and 2 look like the same camera or enclosure (same background)? Compare with row 3.
5. **(Wrong only) Why did row 2 beat row 3?** Similar pose, lighting or background in row 2? Or is row 3 just a bad view of the true bear?

## 0. Imports and setup

In [ ]:
import os
import sys
from pathlib import Path

# Run from the repo folder (the first folder up that has requirements.txt)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image as ShowImage, display

from experiments import heatmaps
from core.preprocess import load_feature_extractor

## 1. Rerun the baseline on every validation video

One row per validation video: the winner (`match_*`), the best video of the true bear (`true_best_*`), the best video of any other bear (`other_best_*`) and the `margin`.

In [ ]:
results, seq_to_emb = heatmaps.match_query_videos(query_split="validation")
results.head()

Accuracy check: should be 88.84% (191/215), the same as `core_objective.py`.

In [ ]:
results["correct"].mean() * 100, results["correct"].sum(), len(results)

## 2. Numbers first: how do correct and wrong videos differ?

Before looking at pictures, three numbers per video (from the saved embeddings, no images needed):
- **margin**: how safe the decision was (see above).
- **frames_right**: share of the 20 frames that, *on their own*, are closest to a video of the true bear. If this is high but the video is still wrong, **averaging the frames hurt**, which is exactly what a learned attention model could fix.
- **effective_frames**: how many frames really count in the similarity (20 = all equal, lower = a few frames dominate).

In [ ]:
frames, embeddings = heatmaps.load_frames()
stats = heatmaps.frame_stats(frames, embeddings, results, seq_to_emb)
stats.groupby("correct")[["margin", "frames_right", "effective_frames", "top_frame_share"]].median()

In [ ]:
fig, path = heatmaps.plot_correct_vs_wrong(stats)
plt.show()

Wrong videos where **at least half of the frames were right on their own**: averaging all 20 frames turned a mostly-right video into a wrong answer.

In [ ]:
wrong = stats[~stats["correct"]]
wrong[wrong["frames_right"] >= 0.5][["sequence", "identity", "predicted", "margin", "frames_right"]]

## 3. Which bears get confused with which?

A pair that shows up many times is worth looking at side by side.

In [ ]:
heatmaps.confusion_pairs(results)

## 4. Pick the examples

- **Correct**: for every bear, its most confident video (largest margin) and its closest call (smallest margin): 26 videos.
- **Wrong**: all 24, sorted from the clearest miss to the nearest miss.

In [ ]:
successes, failures = heatmaps.pick_examples(results)
successes[["sequence", "identity", "kind", "margin", "match_sequence", "other_best_identity"]]

In [ ]:
failures[["sequence", "identity", "predicted", "margin", "match_sequence", "true_best_sequence"]]

## 5. Load the model and verify the heatmap math

In [ ]:
model, transform = load_feature_extractor(device="cpu")
type(model).__name__

### Check 1: the patch scores add up exactly to the decision

For 4 example videos, and for all 3 sections of their figures, everything is recomputed from the images:
- `embedding_vs_grid_mean`: model output vs the average of its 7 x 7 patch grid. About 0 means the frame embedding really is the plain average of the 49 regions.
- `saved_vs_recomputed`: the saved embeddings vs a fresh run. About 0 means the figures use the same numbers as the baseline.
- `cosine_from_patches` vs `cosine_baseline`: all 20 x 49 = 980 patch scores added up and rescaled, vs the similarity the baseline used to decide. `cosine_diff` about 0 means the heatmap splits the real decision exactly.
- `share_diff`: frame %s from the saved embeddings vs from the patch scores (about 0).
- `negative_patches`: share of patch scores below 0, which the heatmap shows as blue.

In [ ]:
check_videos = [successes["sequence"].iloc[0], successes["sequence"].iloc[1],
                failures["sequence"].iloc[0], failures["sequence"].iloc[-1]]
checks = heatmaps.verify_heatmap_math(results, frames, embeddings, seq_to_emb, model, transform, check_videos)
checks

In [ ]:
checks[["embedding_vs_grid_mean", "saved_vs_recomputed", "cosine_diff", "share_diff"]].max()

### Check 2: the heatmap is drawn in the right place

For each of the 49 cells, cover that cell's area of the original 256 x 128 frame with random noise and see which of the 49 patch features changes most. If the heatmap is placed correctly (no flip, no transpose, right stretch), the covered cell itself should change most. The table compares the layout we draw with flipped and transposed layouts: only "as drawn" should stand out.

Neighbors also change a bit, because Swin's shifted windows mix each cell with its surroundings. So a hot cell means "this region and its immediate surroundings", not a sharp box.

In [ ]:
orientation_frames = [frames[frames["sequence"] == s].iloc[10]["member"] for s in check_videos[:3]]
heatmaps.check_grid_orientation(model, transform, orientation_frames)

## 6. Make all the figures

Saved to `results/heatmaps/` as `success_confident_*.jpg`, `success_close_*.jpg` and `failure_*.jpg` (all 20 frames per video, so they are tall).

In [ ]:
paths = {}
for _, example in pd.concat([successes, failures]).iterrows():
    fig, path = heatmaps.plot_example(example, frames, embeddings, seq_to_emb, model, transform)
    plt.close(fig)
    paths[example["sequence"]] = path
len(paths)

## 7. Correct: confident

What a *good* match looks like. Use these as the reference when judging the others.

In [ ]:
for seq in successes.loc[successes["kind"] == "confident", "sequence"]:
    display(ShowImage(paths[seq]))

## 8. Correct: close calls

Right answer, but row 3 (a wrong bear) was almost as similar. What nearly went wrong?

In [ ]:
for seq in successes.loc[successes["kind"] == "close call", "sequence"]:
    display(ShowImage(paths[seq]))

## 9. Wrong

Sorted from the clearest miss to the nearest miss. Compare row 2 (the wrong bear that won) with row 3 (the true bear that lost).

In [ ]:
for seq in failures["sequence"]:
    display(ShowImage(paths[seq]))

## 10. Top-patch gallery: what parts does the model rely on?

The heatmaps are coarse and colored, so here we cut out only the **single best patch of each validation video** and show it as a **plain image** (no colors), enlarged with one cell of context around it. Under each tile is the whole frame with an orange box showing where the patch came from.

How a video's top patch is chosen: score all 20 frames x 49 patches toward the video it was matched to, take the best patch of each frame, and keep the 3 best frames. The gallery shows rank 1.

This scores every validation video (about 15 minutes on a CPU), so the result is saved to `results/heatmaps/top_patches.csv` and reused next time.

In [ ]:
patches_csv = "results/heatmaps/top_patches.csv"
if os.path.exists(patches_csv):
    patches = pd.read_csv(patches_csv, dtype={"identity": str, "predicted": str})
else:
    patches = heatmaps.top_patches(results, frames, seq_to_emb, model, transform, k=3)
    patches.to_csv(patches_csv, index=False)
patches.head()

### Correct videos (sample from all 191)

Up to 4 videos per bear, picked at random (fixed seed). What does a successful match rely on: head, back, legs, fur texture, or background?

In [ ]:
top1 = patches[patches["rank"] == 1]
correct_tiles = (top1[top1["correct"] & (top1["target"] == "winner")]
                 .sample(frac=1, random_state=0)      # shuffle, then keep up to 4 per bear
                 .groupby("identity").head(4)
                 .sort_values("identity"))
fig, path = heatmaps.plot_patch_gallery(correct_tiles, f"Top patch of correct videos ({len(correct_tiles)} of 191)",
                                        "results/heatmaps/gallery_correct.png")
plt.show()

### Wrong videos (all 24): what matched the WRONG bear

Compare with the correct gallery. Darker, blurrier, more background, or a different body part?

In [ ]:
wrong_tiles = top1[~top1["correct"] & (top1["target"] == "winner")].sort_values(["identity", "predicted"])
fig, path = heatmaps.plot_patch_gallery(wrong_tiles, "Top patch of wrong videos: what matched the wrong bear (all 24)",
                                        "results/heatmaps/gallery_wrong.png")
plt.show()

### Wrong videos (all 24): what matched the TRUE bear

The same 24 videos, but the best patch toward the closest video of the right bear (which lost). Is it the same patch as above, or a different part of the bear?

In [ ]:
true_tiles = top1[~top1["correct"] & (top1["target"] == "true_bear")].sort_values(["identity", "predicted"])
fig, path = heatmaps.plot_patch_gallery(true_tiles, "Same wrong videos: what matched the TRUE bear (all 24)",
                                        "results/heatmaps/gallery_wrong_true_bear.png")
plt.show()

## 11. Notes table

Fill this in while looking at the figures (one row per video), then count the patterns. The columns follow the questions at the top.

In [ ]:
notes = pd.concat([successes, failures])[["sequence", "identity", "predicted", "kind", "margin"]].assign(
    heat_on="",        # bear / background / both
    body_part="",      # head / back / legs / whole body
    weak_frames="",    # why low-% frames are weak: small, dark, blurry, cut off, facing away
    same_place="",     # do rows 1 and 2 share the background? yes / no
    note="",
).reset_index(drop=True)
notes.to_csv("results/heatmaps/notes_template.csv", index=False)
notes.head()